# Cluster Analysis & Restaurant Profiling

In this notebook, I will analyze the final restaurant clusters obtained from the K-Means model to understand their characteristics and identify meaningful differences between the groups.

I will compare the clusters using numerical features, cuisine composition, and restaurant characteristics to develop interpretable profiles of each group.

The objective is to understand what distinguishes the clusters and translate the clustering results into meaningful insights about the restaurants.

## 1. Imports, Setup & Data Loading

I will import the required libraries, set up the project paths, and load the final restaurant dataset containing the cluster assignments generated in the previous notebook.

This dataset will serve as the basis for the cluster analysis.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Project Setup
project_root = Path.cwd().parent
processed_data_path = project_root / "1. data" / "processed"

# Loading Final Dataset
clustered_restaurants = pd.read_csv(
    processed_data_path / "final_restaurant_clusters.csv"
)
display(clustered_restaurants.head())

print("Dataset Shape: ", clustered_restaurants.shape)
print("Columns: ", clustered_restaurants.columns.tolist())
print("Clusters: ", sorted(clustered_restaurants["Clusters"].unique()))

,Restaurant,Cost,Collections,Cuisines,avg_rating,rating_std,median_rl,avg_pic,Clusters
0,Beyond Flavours,800,"Food Hygiene Rated Restaurants in Hyderabad, C...","Chinese, Continental, Kebab, European, South I...",4.28,1.239746,32.0,0.39,1
1,Paradise,800,Hyderabad's Hottest,"Biryani, North Indian, Chinese",4.70,0.771984,28.0,0.02,1
2,Flechazo,1300,"Great Buffets, Hyderabad's Hottest","Asian, Mediterranean, North Indian, Desserts",4.66,0.878704,30.5,0.55,1
3,Shah Ghouse Hotel & Restaurant,800,Late Night Restaurants,"Biryani, North Indian, Chinese, Seafood, Bever...",3.21,1.603626,32.5,0.30,0
4,Over The Moon Brew Company,1200,"Best Bars & Pubs, Food Hygiene Rated Restauran...","Asian, Continental, North Indian, Chinese, Med...",4.34,1.065719,36.0,0.44,1


Dataset Shape:  (105, 9)
Columns:  ['Restaurant', 'Cost', 'Collections', 'Cuisines', 'avg_rating', 'rating_std', 'median_rl', 'avg_pic', 'Clusters']
Clusters:  [np.int64(0), np.int64(1)]


## 2. Overall Cluster Composition

I will examine the number and proportion of restaurants assigned to each cluster to understand the overall distribution of the final groups.

In [2]:
# Overall Cluster Distribution
cluster_distribution = (
    clustered_restaurants["Clusters"]
    .value_counts()
    .sort_index()
)

cluster_distribution = pd.to_numeric(cluster_distribution)

cluster_summary = pd.DataFrame({
    "Restaurant_Count": cluster_distribution,
    "Percentage": (
        cluster_distribution.astype(float)
        / cluster_distribution.sum()
        * 100
    ).round(2)
})

display(cluster_summary)

,Restaurant_Count,Percentage
Clusters,,
0,55,52.38
1,50,47.62


## 3. Numerical Cluster Profiling

I will compare the average numerical feature values across the two clusters to understand how they differ in terms of restaurant cost, customer ratings, rating variability, review length, and picture engagement.

These comparisons will help identify the numerical characteristics associated with each cluster.

In [5]:
# Numerical Profiles
numerical_profile = clustered_restaurants.groupby("Clusters").agg({
    "Cost" : "mean",
    "avg_rating" : "mean",
    "rating_std" : "mean",
    "median_rl": "mean",
    "avg_pic": "mean",
    "Restaurant" : "count"
}).rename(columns={"Restaurant" : "restaunrant_count"}).round(2)
display(numerical_profile)

,Cost,avg_rating,rating_std,median_rl,avg_pic,restaunrant_count
Clusters,,,,,,
0,533.64,3.33,1.53,23.7,0.32,55
1,1222.00,3.88,1.22,44.1,1.22,50


### 3.1 Numerical Cluster Profile Interpretation

The numerical profiles reveal clear differences between the two clusters.

Cluster 0 consists of relatively lower-cost restaurants, with a lower average rating, greater rating variability, shorter customer reviews, and fewer pictures per review. These characteristics suggest comparatively lower customer engagement and more varied rating experiences.

Cluster 1 contains relatively higher-cost restaurants with higher average ratings, lower rating variability, longer customer reviews, and more pictures per review. This indicates comparatively stronger customer engagement and more consistent rating patterns.

Although the clusters are relatively balanced in size, they differ noticeably across all the numerical features examined. Overall, the clustering appears to distinguish restaurants based on a combination of cost, customer ratings, and review engagement characteristics.

## 4. Cuisine Composition across Clusters

I will examine the cuisine distribution within each cluster to identify the most common cuisines and determine whether cuisine composition contributes to the differences between the restaurant groups.

Since restaurants can offer multiple cuisines, the distributions will reflect cuisine presence rather than mutually exclusive categories.

In [6]:
# Cuisine Distribution across Clusters
for cluster_id in sorted(clustered_restaurants["Clusters"].unique()):
    cluster_data = clustered_restaurants[
        clustered_restaurants["Clusters"] == cluster_id
    ]
    cuisine_counts = (
        cluster_data["Cuisines"]
        .dropna()
        .str.split(",")
        .explode()
        .str.strip()
    )
    cuisine_counts = cuisine_counts[cuisine_counts != ""].value_counts()
    cuisine_profile = pd.DataFrame({
        "Restaurant_Count": cuisine_counts,
        "Percentage" : (
            cuisine_counts / len(cluster_data) * 100
        ).round(2)
    }).head(10)
    print(f"\nCluster {cluster_id} ({len(cluster_data)} restaurants.)")
    display(cuisine_profile)


Cluster 0 (55 restaurants.)


,Restaurant_Count,Percentage
Cuisines,,
North Indian,31,56.36
Chinese,25,45.45
Biryani,15,27.27
Fast Food,12,21.82
Desserts,9,16.36
South Indian,6,10.91
Andhra,5,9.09
Burger,4,7.27
Beverages,3,5.45



Cluster 1 (50 restaurants.)


,Restaurant_Count,Percentage
Cuisines,,
North Indian,30,60.0
Chinese,18,36.0
Continental,18,36.0
Asian,12,24.0
Italian,12,24.0
Kebab,5,10.0
Mediterranean,5,10.0
European,4,8.0
Desserts,4,8.0


### 4.1 Cuisine Cluster Interpretation 

The cuisine distributions show considerable overlap between the two clusters, with North Indian and Chinese cuisines being prevalent in both groups.

Cluster 0 has a greater representation of Biryani and Fast Food restaurants, alongside predominantly North Indian and Chinese cuisines. This is consistent with its lower average cost profile.

Cluster 1 exhibits a greater presence of Continental, Asian, Italian, and Mediterranean cuisines, alongside North Indian and Chinese restaurants. This reflects a more diverse cuisine composition among the relatively higher-cost restaurants.

Overall, cuisine composition provides additional context for understanding the clusters, but the groups are not exclusively cuisine-based. Their differences appear to reflect a combination of restaurant cost, customer ratings, engagement characteristics, and cuisine offerings.

## 5. Representative Restaurant Analysis

I will examine selected restaurants from each cluster to better understand the characteristics of the restaurants within each group.

By identifying highly rated and lower-rated restaurants, I can observe how the cluster profiles are reflected in individual restaurants and assess the diversity within each group.

In [8]:
# Examining Representative Restaurants
for cluster_id in sorted(clustered_restaurants["Clusters"].unique()):
    cluster_data = clustered_restaurants[
        clustered_restaurants["Clusters"] == cluster_id
    ].dropna(subset=["avg_rating"])
    top_restros = cluster_data.nlargest(3,"avg_rating")
    bottom_restros = cluster_data.nsmallest(3,"avg_rating")
    restaurant_ex = pd.concat(
        [top_restros, bottom_restros]
    ).drop_duplicates(subset=["Restaurant"])
    print(f"\nCluster {cluster_id}")
    display(
        restaurant_ex[
            ["Restaurant", "Cost", "avg_rating", "Cuisines"]
        ].sort_values("avg_rating", ascending=False))


Cluster 0


,Restaurant,Cost,avg_rating,Cuisines
48,Udipi's Upahar,500,4.030,"South Indian, North Indian, Chinese"
59,KS Bakers,200,3.965,"Bakery, Desserts, Fast Food"
83,GD's,500,3.960,"Burger, American"
19,Pakwaan Grand,400,2.710,"North Indian, Chinese, Biryani"
54,Asian Meal Box,200,2.580,Asian
17,Hotel Zara Hi-Fi,400,2.400,"Chinese, North Indian"



Cluster 1


,Restaurant,Cost,avg_rating,Cuisines
14,AB's - Absolute Barbecues,1500,4.880,"European, Mediterranean, North Indian"
22,B-Dubs,1600,4.810,American
27,"3B's - Buddies, Bar & Barbecue",1100,4.760,"North Indian, Mediterranean, European"
31,Pista House,1000,3.230,"Bakery, North Indian, Mughlai, Juices, Chinese"
46,Ulavacharu,1400,3.225,"Andhra, North Indian, Chinese"
44,La La Land - Bar & Kitchen,1000,3.190,"Finger Food, North Indian, Kebab, Chinese"


### 5.1 Representative Restaurant Interpretation

The representative restaurants provide additional context for understanding the two clusters.

Cluster 0 contains a diverse range of lower-cost restaurants, including highly rated establishments such as Udipi's Upahar and KS Bakers, as well as lower-rated restaurants such as Pakwaan Grand and Hotel Zara Hi-Fi. This indicates that the cluster is not defined solely by ratings, but reflects broader characteristics related to cost and customer engagement.

Cluster 1 includes highly rated restaurants such as AB's - Absolute Barbecues, B-Dubs, and 3B's - Buddies, Bar & Barbecue, alongside restaurants with comparatively lower ratings, such as Pista House and Ulavacharu. Despite this variation, the group generally exhibits higher costs, ratings, and customer engagement characteristics.

Overall, the restaurant examples support the numerical profiles identified earlier while highlighting that considerable variation exists within both clusters. The clusters represent broader restaurant characteristics rather than strict categories of quality or cuisine.